In [235]:
from pathlib import Path
import pandas as pd
import numpy as np

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

print("Dataset shape:", df.shape)
print(df["split"].value_counts())

Dataset shape: (12515, 24)
split
train         8789
test          1866
validation    1860
Name: count, dtype: int64


In [236]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

all_vital_features = spo2_hr_features + [
    "respiratory_rate",
    "rr_lag_1",
    "rr_rolling_mean_5",
    "mean_arterial_pressure",
    "map_lag_1",
    "map_rolling_mean_5"
]

target = "future_spo2_5min"

In [237]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

train_ready = train_df.dropna(
    subset=[target]
).copy()

common_val_df = val_df.dropna(
    subset=["spo2", target]
).copy()

print("Training rows:", len(train_ready))
print("Validation rows:", len(common_val_df))

Training rows: 8310
Validation rows: 1748


In [238]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [239]:
def evaluate_gradient_boosting(
    feature_list,
    model_name
):

    X_train = train_ready[feature_list]
    y_train = train_ready[target]

    X_val = common_val_df[feature_list]
    y_val = common_val_df[target]

    model = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                GradientBoostingRegressor(
                    n_estimators=200,
                    learning_rate=0.05,
                    max_depth=3,
                    min_samples_leaf=3,
                    random_state=42
                )
            )
        ]
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            predictions
        )
    )

    r2 = r2_score(
        y_val,
        predictions
    )

    result = {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    return model, predictions, result

In [240]:
gb_spo2_model, gb_spo2_pred, gb_spo2_result = (
    evaluate_gradient_boosting(
        spo2_features,
        "Gradient Boosting - SpO2 History"
    )
)

gb_spo2_result

{'Model': 'Gradient Boosting - SpO2 History',
 'MAE': 0.4317459776081171,
 'RMSE': np.float64(1.3610064405581483),
 'R2': 0.12112765988806407}

In [241]:
gb_hr_model, gb_hr_pred, gb_hr_result = (
    evaluate_gradient_boosting(
        spo2_hr_features,
        "Gradient Boosting - SpO2 + HR"
    )
)

gb_hr_result

{'Model': 'Gradient Boosting - SpO2 + HR',
 'MAE': 0.4415402937088091,
 'RMSE': np.float64(1.3489870940530584),
 'R2': 0.13658214520157863}

In [242]:
gb_all_model, gb_all_pred, gb_all_result = (
    evaluate_gradient_boosting(
        all_vital_features,
        "Gradient Boosting - All Vitals"
    )
)

gb_all_result

{'Model': 'Gradient Boosting - All Vitals',
 'MAE': 0.598006063004987,
 'RMSE': np.float64(1.6150484817973947),
 'R2': -0.23758935030754524}

In [243]:
gb_results = pd.DataFrame([
    gb_spo2_result,
    gb_hr_result,
    gb_all_result
])

gb_results

,Model,MAE,RMSE,R2
0,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
1,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
2,Gradient Boosting - All Vitals,0.598006,1.615048,-0.237589


In [244]:
gb_results.sort_values("MAE")

,Model,MAE,RMSE,R2
0,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
1,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
2,Gradient Boosting - All Vitals,0.598006,1.615048,-0.237589


In [245]:
gb_results.sort_values("RMSE")

,Model,MAE,RMSE,R2
1,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
0,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
2,Gradient Boosting - All Vitals,0.598006,1.615048,-0.237589


In [246]:
all_results.sort_values(
    "R2",
    ascending=False
)

NameError: name 'all_results' is not defined

In [ ]:
gb_results = pd.DataFrame([
    gb_spo2_result,
    gb_hr_result,
    gb_all_result
])

gb_results

In [ ]:
results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

previous_results = pd.read_csv(results_file)

previous_results

In [ ]:
previous_results = previous_results[
    ~previous_results["Model"]
    .str.startswith("Gradient Boosting")
].copy()

In [ ]:
previous_results["Model"].value_counts()

In [ ]:
all_results = pd.concat(
    [
        previous_results,
        gb_results
    ],
    ignore_index=True
)

all_results = (
    all_results
    .drop_duplicates(
        subset=["Model"],
        keep="last"
    )
    .reset_index(drop=True)
)

all_results

In [ ]:
print("Rows:", len(all_results))
print(
    "Unique models:",
    all_results["Model"].nunique()
)

In [ ]:
all_results["Model"].value_counts()

In [ ]:
all_results.sort_values("MAE")

In [ ]:
all_results.sort_values("RMSE")

In [ ]:
all_results.sort_values(
    "R2",
    ascending=False
)

In [247]:
all_results.to_csv(
    results_file,
    index=False
)

print("All 11 model results saved!")

NameError: name 'all_results' is not defined

In [ ]:
check_results = pd.read_csv(results_file)

print("Saved rows:", len(check_results))
print(
    "Saved unique models:",
    check_results["Model"].nunique()
)

In [248]:
print("gb_spo2_result exists:", "gb_spo2_result" in globals())
print("gb_hr_result exists:", "gb_hr_result" in globals())
print("gb_all_result exists:", "gb_all_result" in globals())

gb_spo2_result exists: True
gb_hr_result exists: True
gb_all_result exists: True


In [249]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

print(results_file)

c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\data\processed\model_results.csv


In [250]:
previous_results = pd.read_csv(results_file)

print("Existing rows:", len(previous_results))
print("Existing unique models:", previous_results["Model"].nunique())

previous_results

Existing rows: 50
Existing unique models: 8


,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668
9,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815


In [251]:
gb_results = pd.DataFrame([
    gb_spo2_result,
    gb_hr_result,
    gb_all_result
])

gb_results

,Model,MAE,RMSE,R2
0,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
1,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
2,Gradient Boosting - All Vitals,0.598006,1.615048,-0.237589


In [252]:
previous_results = previous_results[
    ~previous_results["Model"]
    .str.startswith("Gradient Boosting")
].copy()

print("Rows before adding Gradient Boosting:", len(previous_results))

Rows before adding Gradient Boosting: 50


In [253]:
all_results = pd.concat(
    [
        previous_results,
        gb_results
    ],
    ignore_index=True
)

all_results = (
    all_results
    .drop_duplicates(
        subset=["Model"],
        keep="last"
    )
    .reset_index(drop=True)
)

all_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
8,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
9,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582


In [254]:
print("Rows:", len(all_results))
print(
    "Unique models:",
    all_results["Model"].nunique()
)

print()
print(all_results["Model"].value_counts())

Rows: 11
Unique models: 11

Model
Persistence Baseline                1
Linear Regression - SpO2 History    1
Ridge - SpO2 History                1
Ridge - SpO2 + HR                   1
Ridge - All Vitals                  1
Random Forest - SpO2 History        1
Random Forest - SpO2 + HR           1
Random Forest - All Vitals          1
Gradient Boosting - SpO2 History    1
Gradient Boosting - SpO2 + HR       1
Gradient Boosting - All Vitals      1
Name: count, dtype: int64


In [255]:
all_results.to_csv(
    results_file,
    index=False
)

print("All 11 model results saved!")

All 11 model results saved!


In [256]:
check_results = pd.read_csv(results_file)

print("Saved rows:", len(check_results))
print(
    "Saved unique models:",
    check_results["Model"].nunique()
)

Saved rows: 11
Saved unique models: 11


In [257]:
check_results.sort_values("MAE")

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
9,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
4,Ridge - All Vitals,0.512447,1.384467,0.090568


In [258]:
check_results.sort_values("RMSE")

,Model,MAE,RMSE,R2
9,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
8,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
10,Gradient Boosting - All Vitals,0.598006,1.615048,-0.237589


In [259]:
check_results.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
9,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
8,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
10,Gradient Boosting - All Vitals,0.598006,1.615048,-0.237589


In [260]:
check_results = pd.read_csv(results_file)

print("Rows:", len(check_results))
print("Unique models:", check_results["Model"].nunique())

check_results.sort_values("MAE")

Rows: 11
Unique models: 11


,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
9,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
4,Ridge - All Vitals,0.512447,1.384467,0.090568
